In [7]:
import sys
import glayout

print("Python:", sys.executable)
print("gLayout:", glayout.__file__)

from glayout.pdk.mappedpdk import MappedPDK
from glayout.pdk.sky130_mapped import sky130_mapped_pdk as sky130
from glayout.pdk.gf180_mapped import gf180_mapped_pdk as gf180

from glayout.primitives.fet import nmos, pmos
from glayout.primitives.via_gen import via_stack
from glayout.primitives.guardring import tapring
from glayout.primitives.mimcap import mimcap

from gdsfactory import Component
from gdsfactory.components import text_freetype, rectangle

Python: /foss/designs/layout/glayout_env/bin/python
gLayout: /foss/designs/layout/gLayout/src/glayout/__init__.py


In [8]:
import os

run_drc_py = (
    "/foss/pdks/ciel/gf180mcu/"
    "versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/"
    "gf180mcuD/libs.tech/klayout/tech/drc/run_drc.py"
)

print("Existe:", os.path.isfile(run_drc_py))
print(run_drc_py)

with open(run_drc_py, "r") as f:
    lines = f.readlines()

for i in range(min(160, len(lines))):
    print(f"{i+1:03d}: {lines[i].rstrip()}")

Existe: True
/foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/drc/run_drc.py
001: ################################################################################################
002: # Copyright 2023 GlobalFoundries PDK Authors
003: #
004: # Licensed under the Apache License, Version 2.0 (the "License");
005: # you may not use this file except in compliance with the License.
006: # You may obtain a copy of the License at
007: #
008: #     https://www.apache.org/licenses/LICENSE-2.0
009: #
010: # Unless required by applicable law or agreed to in writing, software
011: # distributed under the License is distributed on an "AS IS" BASIS,
012: # WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
013: # See the License for the specific language governing permissions and
014: # limitations under the License.
015: ################################################################################################
016

In [11]:
from pathlib import Path

# 1. Definimos la carpeta exacta de tu proyecto OpenFASOC en el entorno Linux
carpeta_proyecto = Path("/foss/designs/layout")

# 2. Definimos la ruta completa del archivo
#ruta_gds = carpeta_proyecto / "test_inverter.gds"


In [12]:
nmos_kwargs = {
    "with_tie": True,
    "with_dnwell": True,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2","met1"),
    "dummy_routes": True
}

pmos_kwargs = {
    "with_tie": True,
    "dnwell": False,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2","met1"),
    "dummy_routes": True
}

mimcap_kwargs = {
    "option": "B",
    "with_extension": True,
    "extension_direction": "S",
}     

In [14]:
import os
import glob
import subprocess
import xml.etree.ElementTree as ET
from glayout.pdk.gf180_mapped import gf180_mapped_pdk as gf180
import time


def imprimir_resumen_drc(ruta_reporte):
    """Parsea el archivo .lyrdb de KLayout e imprime un resumen limpio de los errores."""
    try:
        tree = ET.parse(ruta_reporte)
        root = tree.getroot()

        # 1. Mapear nombres de categoría a sus descripciones
        descripciones = {}
        for cat in root.findall(".//category"):
            nombre = cat.find("name")
            desc = cat.find("description")
            if nombre is not None and nombre.text:
                descripciones[nombre.text.strip("'\"")] = (
                    desc.text if desc is not None else ""
                )

        # 2. Contar violaciones reales por categoría
        conteo_errores = {}
        total_errores = 0

        for item in root.findall(".//item"):
            cat_item = item.find("category")

            if cat_item is not None and cat_item.text:
                cat_nombre = cat_item.text.strip("'\"")

                conteo_errores[cat_nombre] = (
                    conteo_errores.get(cat_nombre, 0) + 1
                )

                total_errores += 1

        # 3. Mostrar resumen en consola
        print("\n" + "=" * 70)
        print(
            f"?? RESUMEN DE ERRORES DRC | "
            f"Total de violaciones: {total_errores}"
        )
        print("=" * 70)

        if total_errores == 0:
            print("  ?? ¡Felicidades! No se encontraron violaciones DRC.")
        else:
            for cat, cantidad in conteo_errores.items():
                desc = descripciones.get(cat, "Sin descripción")

                print(
                    f" ? [{cat}] "
                    f"({cantidad} error{'es' if cantidad > 1 else ''}): "
                    f"{desc}"
                )

        print("=" * 70 + "\n")

    except Exception as e:
        print(
            f"?? No se pudo procesar el resumen XML del reporte: {e}"
        )


def run_drc_v2(ruta_gds):

    ruta_gds_str = str(ruta_gds)

    if not os.path.exists(ruta_gds_str):
        print(f"? Error: No existe el archivo {ruta_gds_str}")
        return

    print(f"?? Ejecutando DRC para: {ruta_gds_str} ...")

    # ==========================================================
    # CAMBIO: usar el DRC OFICIAL DEL GF180MCU
    # ==========================================================

    gf180_root = (
        "/foss/pdks/ciel/gf180mcu/versions/"
        "7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD"
    )

    run_drc_py = os.path.join(
        gf180_root,
        "libs.tech",
        "klayout",
        "tech",
        "drc",
        "run_drc.py"
    )

    dir_gds = os.path.dirname(os.path.abspath(ruta_gds_str))

    comando = [
        "python3",
        run_drc_py,
        f"--path={ruta_gds_str}",
        "--variant=B",
        f"--run_dir={dir_gds}",
        "--verbose"
    ]

    resultado = subprocess.run(
        comando,
        capture_output=True,
        text=True
    )

    es_clean = resultado.returncode == 0

    print(
        f"¿Diseño libre de errores (DRC Clean)?: {es_clean}"
    )

    # Mostrar salida del DRC oficial si existe
    if resultado.stdout:
        print("\n----- DRC STDOUT -----")
        print(resultado.stdout)

    if resultado.stderr:
        print("\n----- DRC STDERR -----")
        print(resultado.stderr)

    # Carpetas donde buscaremos el reporte
    dir_actual = os.getcwd()
    dir_foss = "/foss/designs/layout"

    carpetas_busqueda = [
        dir_foss,
        dir_gds,
        dir_actual,
        "/tmp"
    ]

    # También incluimos el directorio donde el GF180
    # puede generar los reportes internamente.
    dir_run_drc = os.path.join(
        gf180_root,
        "libs.tech",
        "klayout",
        "tech",
        "macros",
        "run_drc_main"
    )

    carpetas_busqueda.append(dir_run_drc)

    # Buscamos archivos .lyrdb, .ylrdb o que contengan '_drcreport'
    archivos_encontrados = []

    for carpeta in carpetas_busqueda:

        if not os.path.isdir(carpeta):
            continue

        for patron in [
            "*_drcreport*",
            "*.lyrdb",
            "*.ylrdb"
        ]:
            ruta_busqueda = os.path.join(
                carpeta,
                patron
            )

            archivos_encontrados.extend(
                glob.glob(ruta_busqueda)
            )

    # Filtramos para asegurarnos de que sean archivos válidos
    archivos_validos = [
        f
        for f in set(archivos_encontrados)
        if os.path.isfile(f)
    ]

    if archivos_validos:

        reporte_reciente = max(
            archivos_validos,
            key=os.path.getmtime
        )

        print(
            f"?? Reporte de DRC detectado en: "
            f"{reporte_reciente}"
        )

        # --- IMPRIMIR RESUMEN SIMPLIFICADO ---
        imprimir_resumen_drc(reporte_reciente)

        os.system("pkill -f klayout")
        time.sleep(1)
        
        print("?? Abriendo KLayout...")

        subprocess.Popen([
            "klayout",
            "-e",
            ruta_gds_str,
            "-m",
            reporte_reciente
        ])

    else:

        print(
            "?? No se encontró ningún reporte de DRC "
            "en las rutas esperadas."
        )

In [15]:
import os
import glob
import subprocess
import time
from pathlib import Path

def verificar_drc_manual(nombre_archivo="tg_layout.gds"):
    ruta_gds = os.path.abspath(nombre_archivo)
    dir_gds = os.path.dirname(ruta_gds)

    if not os.path.exists(ruta_gds):
        print(f"? Error: No existe el archivo {ruta_gds}")
        print("Asegúrate de haberlo guardado desde KLayout.")
        return

    print(f"?? Ejecutando DRC para: {ruta_gds} ...")

    # Rutas del PDK GF180
    gf180_root = "/foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD"
    run_drc_py = os.path.join(gf180_root, "libs.tech", "klayout", "tech", "drc", "run_drc.py")

    # Comando de ejecución
    comando = [
        "python3",
        run_drc_py,
        f"--path={ruta_gds}",
        "--variant=B",
        f"--run_dir={dir_gds}"
    ]

    # Ejecutar en segundo plano esperando el resultado
    resultado = subprocess.run(comando, capture_output=True, text=True)
    
    if resultado.returncode == 0:
         print("? DRC Terminado con éxito (el script corrió bien).")
    else:
         print("?? DRC finalizó con errores de ejecución (Revisar logs).")

    # Buscar el archivo .lyrdb recién generado
    archivos_encontrados = glob.glob(os.path.join(dir_gds, "*.lyrdb")) + \
                           glob.glob(os.path.join(dir_gds, "*_drcreport*"))
                           
    archivos_validos = [f for f in set(archivos_encontrados) if os.path.isfile(f)]

    if archivos_validos:
        # Tomar el más reciente
        reporte_reciente = max(archivos_validos, key=os.path.getmtime)
        print(f"?? Reporte de DRC detectado en: {reporte_reciente}")

        # Cerrar KLayout viejo para evitar el problema del sufijo $2
        print("?? Limpiando memoria de KLayout...")
        os.system("pkill -f klayout")
        time.sleep(1) # Pausa breve para asegurar el cierre

        # Abrir KLayout nuevo con el diseño editable y los marcadores listos
        print("?? Abriendo KLayout...")
        subprocess.Popen([
            "klayout",
            "-e",          # Modo editable
            ruta_gds,      # Tu archivo modificado
            "-m",          # Cargar marcadores
            reporte_reciente
        ])
    else:
        print("? No se generó ningún archivo .lyrdb de marcadores.")

In [37]:

def _obtener_puerto_bot_m4(ref, direccion: str):
    """Obtiene el puerto de Metal 4 (Bottom Plate) del componente."""
    nombres_m4 = [
        f"bottom_met_{direccion}",
        f"bot_met_{direccion}",
        f"met4_{direccion}",
        f"bottom_met4_{direccion}",
    ]
    for n in nombres_m4:
        if n in ref.ports:
            return ref.ports[n]
    for name, port in ref.ports.items():
        if "met4" in name and direccion in name:
            return port
        if "bottom" in name and "via" not in name and direccion in name:
            return port
    return None


def pga_cf_bank(
    pdk: MappedPDK = gf180,
    width: float = 11.0,
    length: float = 11.0,
    multiplicadores: list[int] = [1, 2, 4, 8, 16],
    option: str = "B",
    espaciado_bloques_x: float = 40.0,
    separacion_ramas_y: float = 90.0,
    separacion_grid: float = 14.0,
    ancho_puente_met5: float = 4.0,
) -> Component:
    """Banco PGA con topología simétrica desacoplada: Top Plate (Met5) y Bottom Plate (Met4 continuo -> Met5 Out)."""
    bank = Component("pga_cf_bank")
    pos_x_actual = 0.0

    for m in multiplicadores:
        # 1. Configuración de Matriz 2D
        filas = int(math.isqrt(m))
        while m % filas != 0 and filas > 1:
            filas -= 1
        cols = m // filas

        grid_p, grid_n = [], []
        contador = 0

        # 2. Instanciación (with_extension=False mantiene el Bottom Plate en Metal 4 puro)
        for r in range(filas):
            fila_p, fila_n = [], []
            for c in range(cols):
                if contador >= m:
                    break

                mim_cell = mimcap(
                    pdk=pdk,
                    size=(width, length),
                    option=option,
                    with_extension=False,
                )

                # Rama P
                ref_p = bank << mim_cell
                ref_p.x = pos_x_actual + c * (width + separacion_grid)
                ref_p.y = -r * (length + separacion_grid)
                fila_p.append(ref_p)

                # Rama N
                ref_n = bank << mim_cell
                ref_n.x = pos_x_actual + c * (width + separacion_grid)
                ref_n.y = -separacion_ramas_y - r * (length + separacion_grid)
                fila_n.append(ref_n)

                contador += 1

            grid_p.append(fila_p)
            grid_n.append(fila_n)

        # 3. Enrutado Ortogonal Simétrico por Capas
        for grid in [grid_p, grid_n]:
            col_centro = len(grid[0]) // 2 if len(grid[0]) > 1 else 0

            # A. TOP PLATE (Metal 5): Conexión Horizontal por Fila
            for r in range(len(grid)):
                for c in range(len(grid[r]) - 1):
                    bank << straight_route(
                        pdk,
                        grid[r][c].ports["top_met_E"],
                        grid[r][c + 1].ports["top_met_W"],
                        glayer1="met5",
                    )

            # B. TOP PLATE (Metal 5): Puente Vertical Delgado en el Canal Central
            if len(grid) > 1:
                for r in range(len(grid) - 1):
                    node_top = grid[r][col_centro]
                    node_bot = grid[r + 1][col_centro]
                    bank << straight_route(
                        pdk,
                        node_top.ports["top_met_S"],
                        node_bot.ports["top_met_N"],
                        glayer1="met5",
                        width=ancho_puente_met5,
                    )

            # C. BOTTOM PLATE (Metal 4): Horizontales y Verticales en Met4 (Sin cortocircuito con Met5)
            for r in range(len(grid)):
                # Malla Horizontal Met4
                for c in range(len(grid[r]) - 1):
                    p_m4_e = _obtener_puerto_bot_m4(grid[r][c], "E")
                    p_m4_w = _obtener_puerto_bot_m4(grid[r][c + 1], "W")
                    if p_m4_e and p_m4_w:
                        bank << straight_route(
                            pdk, p_m4_e, p_m4_w, glayer1="met4"
                        )

                # Conexión Vertical Met4 entre filas (Pasa por debajo del Met5)
                if r < len(grid) - 1:
                    for c_idx in range(len(grid[r])):
                        p_m4_s = _obtener_puerto_bot_m4(grid[r][c_idx], "S")
                        p_m4_n = _obtener_puerto_bot_m4(grid[r + 1][c_idx], "N")
                        if p_m4_s and p_m4_n:
                            bank << straight_route(
                                pdk, p_m4_s, p_m4_n, glayer1="met4"
                            )

            # D. SALIDA (out): Conversión de Metal 4 a Metal 5 mediante Via Stack
            last_row_center = grid[-1][col_centro]
            p_m4_out = _obtener_puerto_bot_m4(last_row_center, "S")

            if p_m4_out:
                v_stack = bank << via_stack(pdk, "met4", "met5")
                v_stack.x = last_row_center.x
                v_stack.y = last_row_center.y - (length / 2.0) - 6.0

                p_via_in = (
                    v_stack.ports["bottom_met_N"]
                    if "bottom_met_N" in v_stack.ports
                    else v_stack.ports["e1_N"]
                )
                if p_via_in:
                    bank << straight_route(
                        pdk, p_m4_out, p_via_in, glayer1="met4"
                    )

        # 4. Puertos Globales
        # Entrada Top Plate (Metal 5 Izquierda)
        bank.add_port(
            f"CfP_x{m}_IN_TOP_W", port=grid_p[0][0].ports["top_met_W"]
        )
        bank.add_port(
            f"CfN_x{m}_IN_TOP_W", port=grid_n[0][0].ports["top_met_W"]
        )

        ancho_bloque = (
            cols * width + (cols - 1) * separacion_grid if cols > 0 else width
        )
        pos_x_actual += ancho_bloque + espaciado_bloques_x

    return bank


if __name__ == "__main__":
    ruta_gds = "/foss/designs/layout/pga_cf_bank.gds"
    layout = pga_cf_bank(gf180, width=11.0, length=11.0, option="B")
    layout.write_gds(ruta_gds)
    print(
        f"Banco PGA generado con enrutado simétrico Met5/Met4 limpio en: {ruta_gds}"
    )
    run_drc_v2(ruta_gds)

/tmp/ipykernel_27822/1483205237.py:163: UserWarning: Unnamed cells, 31 in 'pga_cf_bank$15'
  layout.write_gds(ruta_gds)
2026-08-24 03:09:53.770 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/foss/designs/layout/pga_cf_bank.gds'


Banco PGA generado con enrutado simétrico Met5/Met4 limpio en: /foss/designs/layout/pga_cf_bank.gds
?? Ejecutando DRC para: /foss/designs/layout/pga_cf_bank.gds ...
¿Diseño libre de errores (DRC Clean)?: False

----- DRC STDOUT -----
2026-08-24 03:09:56 +0200: Memory Usage (451728K) : Starting running GF180MCU Klayout DRC runset on /foss/designs/layout/pga_cf_bank.gds
2026-08-24 03:09:56 +0200: Memory Usage (451728K) : Ruby Version for klayout: 3.2.3


----- DRC STDERR -----
24-Aug-2026 03:09:54 | INFO    | Your Klayout version is: KLayout 0.30.8
24-Aug-2026 03:09:54 | INFO    | ## Generating template with for the following rule tables: ['dummy_metal4.drc', 'dummy_exclude.drc', 'nat_split.drc', 'dualgate.drc', 'drc_bjt.drc', 'dummy_comp.drc', 'nplus.drc', 'nwell.drc', 'dummy_metal1.drc', 'esd.drc', 'via4.drc', 'metal2.drc', 'via4_split.drc', 'sram_3p3.drc', 'via1.drc', 'dnwell.drc', 'via3.drc', 'nat.drc', 'metaltop_30k.drc', 'dummy_metal2.drc', 'sab.drc', 'dummy_poly2.drc', 'metal1.drc

ERROR: Marker database has unknown format


(22, 0)
## gf180mcu PDK Pcells loaded.
['/foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/pymacros', '/foss/tools/klayout/pymod', '/foss/tools/klayout_gdsfactory9/lib/python3.12/site-packages', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '/headless/.local/lib/python3.12/site-packages', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/headless/.klayout/python', '/headless/.klayout/salt/klive/python', '/headless/.klayout/salt/KLayoutPluginUtils/python', '/headless/.klayout/salt/gdsfactory/python']
klive 0.4.1 is running
